# Importy 

In [2]:
import torch 
import torch.nn as nn  
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import plotly.graph_objects as go 

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

seed = 123
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


# Wyświetlanie obrazków

Tworzymy funkcje do wyświetlania obrazków

In [3]:
def showimg(img):
    img = img / 2 + 0.5 
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1,2,0)))
    plt.show()


# Ładowanie danych 

Tworzymy funkcje, która importuje dane za pomocą biblioteki torchvision. Zbiór danych to CIFAR10, który zawiera zdjęcia RGB dziesięciu różnych rzeczy. 

In [4]:
def data_load(batch_size, show = False):
    transform = transforms.Compose([
        transforms.ToTensor(), #[0,255] na [0,1]
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
    ])
    trainset = torchvision.datasets.CIFAR10(root = './data', train = True, download = True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root = './data', train = False, download = True, transform=transform)
    
    trainload = torch.utils.data.DataLoader(trainset,batch_size = batch_size,shuffle = True, num_workers = 4, pin_memory=True, persistent_workers=True )
    testload = torch.utils.data.DataLoader(testset,batch_size = batch_size,shuffle = False, num_workers = 4, pin_memory=True, persistent_workers=True )

    classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
    if show:
        dataiter = iter(trainload)
        images, labels = next(dataiter)
        showimg(torchvision.utils.make_grid(images))
        for j in range(batch_size):
            print(classes[labels[j]], end=' ')

    return ((trainset,trainload), (testset,testload), classes)


# Pierwszy typ sieci 

Na początku stworzymy prosty model sieci splotowej oparty na architekturze LeNet. 

In [5]:
class cnn(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5) # filtr 5x5 
        self.pool = nn.MaxPool2d(2,2) 
        self.conv2 = nn.Conv2d(6,16,5) # filtr 5x5
        self.fc1 = nn.Linear(16*5*5,120) #16 warstw 5x5 (bo uzyjemy w forward poola)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x,1) # splaszczenie do liniowych(wyjatek batch)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x 


Teraz tworzymy funkcje, która będzie testowała poprawność modelu

In [6]:
def test(testload,net):
    corr = 0
    total = 0
    net.eval()
    with torch.no_grad():
        for data in testload:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = net(images)
            _, pred = torch.max(outputs.data, 1)

            total += labels.size(0) 
            corr += (pred == labels).sum().item()
    acc = 100 * corr / total
    return(acc)


Oraz funkcje do trenowania naszego modelu, z opcją walidacji danych, tak abyśmy mieli wgląd do wyników

In [7]:
def train(trainload, net, crit, optim, testload = [], validate = True, epochs = 10):
    h_train_acc = []
    h_test_acc = []
    for epoch in range(epochs):
        net.train()
        r_loss = 0
        corr_train = 0
        total_train = 0
        for i, data in enumerate(trainload):
            optim.zero_grad() # zerujemy gradienty
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = net(inputs)
            loss = crit(outputs, labels) # f straty
            loss.backward() # obliczmy grad dla wszystkich parametrow
            optim.step()
            r_loss += loss.item()
            _, pred = torch.max(outputs.data,1)
            total_train += labels.size(0)
            corr_train += (pred==labels).sum().item()
            
        loss_epoch = r_loss/len(trainload)
        train_acc_epoch = 100* corr_train/total_train
        print(f"Epoch {epoch+1}| Loss: {loss_epoch} | Train Acc: {train_acc_epoch}% ", end="")
        if validate: 
            test_acc_epoch = test(testload, net)
            print(f"| Test Acc: {test_acc_epoch}%")
        h_train_acc.append(train_acc_epoch)
        h_test_acc.append(test_acc_epoch)
    print('finished')
    
    return(h_train_acc, h_test_acc)


In [8]:
(trainset,trainload), (testset,testload), classes = data_load(128)
net = cnn()
crit = nn.CrossEntropyLoss()
optim = torch.optim.Adam(net.parameters(), lr=0.001)
net.to(device)
h_train, h_test = train(trainload,net,crit,optim,testload)


Files already downloaded and verified
Files already downloaded and verified
Epoch 1| Loss: 1.7238572555429794 | Train Acc: 35.896% | Test Acc: 43.5%
Epoch 2| Loss: 1.431461331484568 | Train Acc: 48.32% | Test Acc: 49.67%
Epoch 3| Loss: 1.3062544064143735 | Train Acc: 53.194% | Test Acc: 54.87%
Epoch 4| Loss: 1.2260461871886192 | Train Acc: 56.392% | Test Acc: 57.16%
Epoch 5| Loss: 1.1618260637573574 | Train Acc: 58.856% | Test Acc: 58.45%
Epoch 6| Loss: 1.1088287487359303 | Train Acc: 60.534% | Test Acc: 60.09%
Epoch 7| Loss: 1.062838186540872 | Train Acc: 62.364% | Test Acc: 61.34%
Epoch 8| Loss: 1.0184627297284352 | Train Acc: 64.074% | Test Acc: 61.42%
Epoch 9| Loss: 0.9847993981807738 | Train Acc: 65.204% | Test Acc: 63.14%
Epoch 10| Loss: 0.9490209813313106 | Train Acc: 66.588% | Test Acc: 63.14%
finished


Teraz stworzymy funkcję do wyświetlania wykresów dla naszych danych. 

In [9]:
def plot(accuracy_train, accuracy_val):
    x=list(range(1,len(accuracy_train)+1))
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=x,
                y=accuracy_train,
                name="Accuracy train",
                line=dict(color="purple")))
    fig.add_trace(
        go.Scatter(x=x,
                y=accuracy_val,
                name="Accuracy val",
                line=dict(color="red")))


    fig.update_layout(title_text="Przebieg procesu uczenia")
    fig.show()
plot(h_train,h_test)


Końcowo otrzymujemy dokładność ok. 63 % na zbiorze testowym. Przeuczenie modelu nie jest widoczne. 


# Druga sieć neuronowa

Teraz stworzymy sieć opartą na architekturze VGG

In [10]:
class vgg(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #32x32 na 16x16
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #16x16 na 8x8
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  #8x8 na 4x4
        )
        
        self.flatten_size = 128 * 4 * 4 
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 512),
            nn.ReLU(),
            nn.Linear(512, 10) 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [12]:

net = vgg()
crit = nn.CrossEntropyLoss()
optim = torch.optim.Adam(net.parameters(), lr=0.001)
net.to(device)
h_train, h_test = train(trainload,net,crit,optim,testload)


Epoch 1| Loss: 1.416659469189851 | Train Acc: 48.74% | Test Acc: 59.04%
Epoch 2| Loss: 0.9820606229860155 | Train Acc: 65.114% | Test Acc: 67.04%
Epoch 3| Loss: 0.7874316906989993 | Train Acc: 72.372% | Test Acc: 72.5%
Epoch 4| Loss: 0.6429851796011181 | Train Acc: 77.304% | Test Acc: 73.42%
Epoch 5| Loss: 0.5353346114115947 | Train Acc: 81.286% | Test Acc: 74.64%
Epoch 6| Loss: 0.4389935141176824 | Train Acc: 84.644% | Test Acc: 75.32%
Epoch 7| Loss: 0.3381302993163428 | Train Acc: 88.04% | Test Acc: 76.59%
Epoch 8| Loss: 0.24898041070193586 | Train Acc: 91.384% | Test Acc: 75.71%
Epoch 9| Loss: 0.18360767738364847 | Train Acc: 93.578% | Test Acc: 75.42%
Epoch 10| Loss: 0.1334199447403936 | Train Acc: 95.466% | Test Acc: 75.75%
finished


In [13]:
plot(h_train, h_test)

Na wykresie widzimy, że osiągneliśmy wyższą dokładność - ok. 75 %. Widzimy także zjawisko overfittingu, nasz model przeuczył się, przez co dokładność na danych testowych była znacząco mniejsza od dokładności na danych treningowych(ok. 95%)

# Batch normalization

Spróbujemy teraz zastosować batch normalization, na powyszej architekturze

In [14]:
class vggBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), 
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #32x32 na 16x16
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #16x16 na 8x8
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  #8x8 na 4x4
        )
        
        self.flatten_size = 128 * 4 * 4 
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 512),
            nn.ReLU(),
            nn.Linear(512, 10) 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [15]:
net = vggBatchNorm()
crit = nn.CrossEntropyLoss()
optim = torch.optim.Adam(net.parameters(), lr=0.001)
net.to(device)
h_train, h_test = train(trainload,net,crit,optim,testload)

Epoch 1| Loss: 1.250282067319621 | Train Acc: 55.054% | Test Acc: 62.11%
Epoch 2| Loss: 0.828540656420276 | Train Acc: 70.588% | Test Acc: 69.55%
Epoch 3| Loss: 0.6852806465095266 | Train Acc: 75.74% | Test Acc: 73.95%
Epoch 4| Loss: 0.5843154887866486 | Train Acc: 79.502% | Test Acc: 73.67%
Epoch 5| Loss: 0.5004724223747887 | Train Acc: 82.498% | Test Acc: 75.1%
Epoch 6| Loss: 0.4245069536863995 | Train Acc: 84.968% | Test Acc: 76.81%
Epoch 7| Loss: 0.35837201762687215 | Train Acc: 87.446% | Test Acc: 76.74%
Epoch 8| Loss: 0.30031155335628773 | Train Acc: 89.616% | Test Acc: 74.92%
Epoch 9| Loss: 0.24899331729887697 | Train Acc: 91.186% | Test Acc: 76.79%
Epoch 10| Loss: 0.20601482954247832 | Train Acc: 92.812% | Test Acc: 79.04%
finished


In [16]:
plot(h_train, h_test)

Widzimy, że dokładność lekko wzrosła do ok. 79%, jednak w dalszym ciągu mamy overfitting. 

# Dropout

Teraz zastosujemy dodatkowo technikę dropout, która powinna zapobiegać przeuczniu modelu. 

In [30]:
class vggBatchNormDrpout(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), 
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #32x32 na 16x16
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #16x16 na 8x8
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  #8x8 na 4x4
        )
        
        self.flatten_size = 128 * 4 * 4 
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10) 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [31]:
net = vggBatchNormDrpout()
crit = nn.CrossEntropyLoss()
optim = torch.optim.Adam(net.parameters(), lr=0.001)
net.to(device)
h_train, h_test = train(trainload,net,crit,optim,testload)

Epoch 1| Loss: 1.3679213743380574 | Train Acc: 50.414% | Test Acc: 61.44%
Epoch 2| Loss: 0.9998073423914897 | Train Acc: 64.622% | Test Acc: 69.92%
Epoch 3| Loss: 0.8611820626746663 | Train Acc: 69.54% | Test Acc: 72.97%
Epoch 4| Loss: 0.7683591396759843 | Train Acc: 73.142% | Test Acc: 71.11%
Epoch 5| Loss: 0.7029260666778935 | Train Acc: 75.43% | Test Acc: 75.34%
Epoch 6| Loss: 0.6399350739501016 | Train Acc: 77.674% | Test Acc: 77.15%
Epoch 7| Loss: 0.5895163555584295 | Train Acc: 79.462% | Test Acc: 78.67%
Epoch 8| Loss: 0.5338516788714377 | Train Acc: 81.214% | Test Acc: 76.06%
Epoch 9| Loss: 0.4937297559302786 | Train Acc: 82.802% | Test Acc: 77.3%
Epoch 10| Loss: 0.4552236744357497 | Train Acc: 83.888% | Test Acc: 77.71%
finished


In [32]:
plot(h_train, h_test)

Widzimy, że poprawność nie uległa poprawie, a nawet pogorszyła się(ok. 78%), natomiast pozbyliśmy się przeuczenia modelu (przynajmniej częsciowo) - teraz dokladnosc na zbiorze treningowym wynosi ok. 84%. Jest to dobra wiadomość, ponieważ przy wiekszej liczbie epok bedziemy w stanie dojść do większej dokładności

In [33]:
net = vggBatchNormDrpout()
crit = nn.CrossEntropyLoss()
optim = torch.optim.Adam(net.parameters(), lr=0.001)
net.to(device)
h_train, h_test = train(trainload,net,crit,optim,testload,epochs=20)

Epoch 1| Loss: 1.352083730118354 | Train Acc: 51.064% | Test Acc: 63.45%
Epoch 2| Loss: 0.9802461440301002 | Train Acc: 65.086% | Test Acc: 68.65%
Epoch 3| Loss: 0.8457831501046105 | Train Acc: 70.306% | Test Acc: 71.47%
Epoch 4| Loss: 0.7574248352014196 | Train Acc: 73.428% | Test Acc: 73.69%
Epoch 5| Loss: 0.6971408783474846 | Train Acc: 75.438% | Test Acc: 76.3%
Epoch 6| Loss: 0.6337316719162495 | Train Acc: 77.854% | Test Acc: 76.03%
Epoch 7| Loss: 0.5755941304556854 | Train Acc: 79.956% | Test Acc: 77.05%
Epoch 8| Loss: 0.5369473254436727 | Train Acc: 80.974% | Test Acc: 76.58%
Epoch 9| Loss: 0.49285012270178635 | Train Acc: 82.562% | Test Acc: 77.13%
Epoch 10| Loss: 0.45208857942115316 | Train Acc: 84.104% | Test Acc: 77.75%
Epoch 11| Loss: 0.4180205606895944 | Train Acc: 85.26% | Test Acc: 79.14%
Epoch 12| Loss: 0.3880504789331075 | Train Acc: 86.284% | Test Acc: 78.53%
Epoch 13| Loss: 0.356592805108146 | Train Acc: 87.336% | Test Acc: 79.88%
Epoch 14| Loss: 0.31911689092588547 

In [34]:
plot(h_train,h_test)

Pzy dwudziestu epokch dostaliśmy dokładność na poziomie ok. 80%, jednak widzimy, że znowu nasz model sie przeucza

# Agumentation

Zastosujemy teraz agumentacje do ładowania danych, to w jeszcze większym stopniu utrudni modelowi przetrenowanie. 

In [47]:
def data_load(batch_size, show = False):
    train_transform = transforms.Compose([
        #transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        #transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(), #[0,255] na [0,1]
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(), #[0,255] na [0,1]
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
    ])
    trainset = torchvision.datasets.CIFAR10(root = './data', train = True, download = True, transform=train_transform)
    testset = torchvision.datasets.CIFAR10(root = './data', train = False, download = True, transform=test_transform)
    
    trainload = torch.utils.data.DataLoader(trainset,batch_size = batch_size,shuffle = True, num_workers = 4, pin_memory=True, persistent_workers=True )
    testload = torch.utils.data.DataLoader(testset,batch_size = batch_size,shuffle = False, num_workers = 4, pin_memory=True, persistent_workers=True )

    classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
    if show:
        dataiter = iter(trainload)
        images, labels = next(dataiter)
        showimg(torchvision.utils.make_grid(images))
        for j in range(batch_size):
            print(classes[labels[j]], end=' ')

    return ((trainset,trainload), (testset,testload), classes)

(trainset,trainload), (testset,testload), classes = data_load(128)




Files already downloaded and verified
Files already downloaded and verified


Dodamy takze zmniejszenie kroku w dalekich epokach

In [42]:
def train(trainload, net, crit, optim, testload = [], validate = True, epochs = 10):
    opt_step = torch.optim.lr_scheduler.StepLR(optim, step_size=10, gamma=0.1)
    h_train_acc = []
    h_test_acc = []
    for epoch in range(epochs):
        net.train()
        r_loss = 0
        corr_train = 0
        total_train = 0
        for i, data in enumerate(trainload):
            optim.zero_grad() # zerujemy gradienty
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = net(inputs)
            loss = crit(outputs, labels) # f straty
            loss.backward() # obliczmy grad dla wszystkich parametrow
            optim.step()
            r_loss += loss.item()
            _, pred = torch.max(outputs.data,1)
            total_train += labels.size(0)
            corr_train += (pred==labels).sum().item()
        opt_step.step()
        loss_epoch = r_loss/len(trainload)
        train_acc_epoch = 100* corr_train/total_train
        print(f"Epoch {epoch+1}| Loss: {loss_epoch} | Train Acc: {train_acc_epoch}% ", end="")
        if validate: 
            test_acc_epoch = test(testload, net)
            print(f"| Test Acc: {test_acc_epoch}%")
        h_train_acc.append(train_acc_epoch)
        h_test_acc.append(test_acc_epoch)
    print('finished')
    
    return(h_train_acc, h_test_acc)


In [48]:
net = vggBatchNormDrpout()
crit = nn.CrossEntropyLoss()
optim = torch.optim.Adam(net.parameters(), lr=0.001)
net.to(device)
h_train, h_test = train(trainload,net,crit,optim,testload,epochs=30)

Epoch 1| Loss: 1.5590938905925702 | Train Acc: 42.544% | Test Acc: 56.33%
Epoch 2| Loss: 1.2281554760530478 | Train Acc: 55.49% | Test Acc: 62.22%
Epoch 3| Loss: 1.084362705955115 | Train Acc: 61.644% | Test Acc: 68.47%
Epoch 4| Loss: 0.9998760227961918 | Train Acc: 64.686% | Test Acc: 69.6%
Epoch 5| Loss: 0.9408908483317441 | Train Acc: 66.98% | Test Acc: 69.3%
Epoch 6| Loss: 0.8967541921169252 | Train Acc: 68.774% | Test Acc: 74.19%
Epoch 7| Loss: 0.8495810864221714 | Train Acc: 70.224% | Test Acc: 69.68%
Epoch 8| Loss: 0.8208990087899406 | Train Acc: 71.314% | Test Acc: 75.22%
Epoch 9| Loss: 0.7918612877731128 | Train Acc: 72.456% | Test Acc: 73.7%
Epoch 10| Loss: 0.76652826098225 | Train Acc: 73.438% | Test Acc: 76.24%
Epoch 11| Loss: 0.6787446043680391 | Train Acc: 76.476% | Test Acc: 79.09%
Epoch 12| Loss: 0.6555475007237681 | Train Acc: 77.078% | Test Acc: 79.38%
Epoch 13| Loss: 0.6399810119815494 | Train Acc: 77.898% | Test Acc: 79.85%
Epoch 14| Loss: 0.6384207639852753 | Train

Agumentacja znacząco utrudniła uczenie się, widzimy ze test acc> train acc, dlatego zmniejszymy teraz dropout w naszym modelu co powinno ułatwić trenowanie i pozwolić uzyskać lepsze wyniki

In [49]:
class vggBatchNormDrpout(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), 
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #32x32 na 16x16
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #16x16 na 8x8
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  #8x8 na 4x4
        )
        
        self.flatten_size = 128 * 4 * 4 
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 10) 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [50]:
net = vggBatchNormDrpout()
crit = nn.CrossEntropyLoss()
optim = torch.optim.Adam(net.parameters(), lr=0.001)
net.to(device)
h_train, h_test = train(trainload,net,crit,optim,testload,epochs=30)

Epoch 1| Loss: 1.4861242362605336 | Train Acc: 45.72% | Test Acc: 56.5%
Epoch 2| Loss: 1.1170205711708654 | Train Acc: 59.97% | Test Acc: 65.34%
Epoch 3| Loss: 0.9734894224749807 | Train Acc: 65.316% | Test Acc: 68.96%
Epoch 4| Loss: 0.897857393602581 | Train Acc: 68.408% | Test Acc: 72.08%
Epoch 5| Loss: 0.8480323967726334 | Train Acc: 70.434% | Test Acc: 73.05%
Epoch 6| Loss: 0.7943579710048178 | Train Acc: 72.138% | Test Acc: 73.85%
Epoch 7| Loss: 0.7568852274161776 | Train Acc: 73.434% | Test Acc: 76.52%
Epoch 8| Loss: 0.7259597393405407 | Train Acc: 74.626% | Test Acc: 76.87%
Epoch 9| Loss: 0.6966742927308582 | Train Acc: 75.77% | Test Acc: 77.37%
Epoch 10| Loss: 0.6810863005078357 | Train Acc: 76.212% | Test Acc: 78.25%
Epoch 11| Loss: 0.5891544825738043 | Train Acc: 79.538% | Test Acc: 81.55%
Epoch 12| Loss: 0.5635137450512108 | Train Acc: 80.322% | Test Acc: 81.67%
Epoch 13| Loss: 0.555162715103925 | Train Acc: 80.842% | Test Acc: 81.74%
Epoch 14| Loss: 0.5500313653360547 | Tra

In [51]:
plot(h_train, h_test)

W dalszym ciągu poprawność na zbiorze testowym jest wieksza od tej na zbiorze treningowym, dlatego poszerzymy sieć o kolejne neurony

In [52]:
class vggBatchNormDrpout(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), 
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #32x32 na 16x16
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #16x16 na 8x8
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  #8x8 na 4x4
        )
        
        self.flatten_size = 256 * 4 * 4 
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 10) 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [53]:
net = vggBatchNormDrpout()
crit = nn.CrossEntropyLoss()
optim = torch.optim.Adam(net.parameters(), lr=0.001)
net.to(device)
h_train, h_test = train(trainload,net,crit,optim,testload,epochs=30)

Epoch 1| Loss: 1.5904243541190692 | Train Acc: 41.954% | Test Acc: 55.73%
Epoch 2| Loss: 1.165536619513236 | Train Acc: 58.002% | Test Acc: 61.23%
Epoch 3| Loss: 1.0063363418859594 | Train Acc: 64.288% | Test Acc: 66.19%
Epoch 4| Loss: 0.9098314325827772 | Train Acc: 67.976% | Test Acc: 72.88%
Epoch 5| Loss: 0.8554572516390125 | Train Acc: 69.958% | Test Acc: 72.35%
Epoch 6| Loss: 0.8039018860863297 | Train Acc: 71.98% | Test Acc: 75.65%
Epoch 7| Loss: 0.7628273131597377 | Train Acc: 73.412% | Test Acc: 76.19%
Epoch 8| Loss: 0.72188698765262 | Train Acc: 74.978% | Test Acc: 75.71%
Epoch 9| Loss: 0.696866165875169 | Train Acc: 75.858% | Test Acc: 76.66%
Epoch 10| Loss: 0.6678473588908115 | Train Acc: 76.958% | Test Acc: 78.98%
Epoch 11| Loss: 0.564785253361363 | Train Acc: 80.448% | Test Acc: 82.28%
Epoch 12| Loss: 0.5421844475409564 | Train Acc: 81.092% | Test Acc: 82.38%
Epoch 13| Loss: 0.5329395415990249 | Train Acc: 81.634% | Test Acc: 82.68%
Epoch 14| Loss: 0.5224995657306193 | Tra

W dalszym ciągu nie mamy przeuczenia, spróbujemy usunąć ostatni maxpooling tak aby do warstw liniowych przekazac wieksza ilość parametrów. Dodatkowo dodajemy drugi filtr 3x3 tak aby sieć miała wgląd na większe detale. Powoduje to oczywiście duży wzrost liczby parametrów, dlatego korzystamy teraz z sgd zamiast adam, ponieważ lepiej sobie z tym radzi.  

In [ ]:
class vgg_hr(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) 
        )
        
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        
        self.flatten_size = 256*8*8  
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 10)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

In [ ]:
net = vgg_hr()
net.to(device)

crit = nn.CrossEntropyLoss()
optim = torch.optim.SGD(net.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
train_acc, test_acc = train(trainload, net, crit, optim, testload, epochs=35)

Epoch 1| Loss: 1.3860597845233615 | Train Acc: 49.324% | Test Acc: 55.64%
Epoch 2| Loss: 0.9369589883043333 | Train Acc: 66.678% | Test Acc: 68.1%
Epoch 3| Loss: 0.7781861901588147 | Train Acc: 72.89% | Test Acc: 75.59%
Epoch 4| Loss: 0.6748905176549311 | Train Acc: 76.622% | Test Acc: 76.97%
Epoch 5| Loss: 0.6074788826505851 | Train Acc: 78.92% | Test Acc: 80.31%
Epoch 6| Loss: 0.5515406175952433 | Train Acc: 80.854% | Test Acc: 80.56%
Epoch 7| Loss: 0.505557365048572 | Train Acc: 82.55% | Test Acc: 81.53%
Epoch 8| Loss: 0.47088123701722423 | Train Acc: 83.86% | Test Acc: 82.67%
Epoch 9| Loss: 0.4428699167488176 | Train Acc: 84.86% | Test Acc: 84.9%
Epoch 10| Loss: 0.41245952477235626 | Train Acc: 85.924% | Test Acc: 80.56%
Epoch 11| Loss: 0.3077661995883183 | Train Acc: 89.574% | Test Acc: 88.6%
Epoch 12| Loss: 0.2779131228356715 | Train Acc: 90.374% | Test Acc: 89.0%
Epoch 13| Loss: 0.2636396003615521 | Train Acc: 90.908% | Test Acc: 89.14%
Epoch 14| Loss: 0.2578312018338372 | Train

In [56]:
plot(train_acc, test_acc)

Widzimy, że dzięki większej ilości parametrów zwiększyliśmy znacząco naszą skutecznoś. Osiągnęliśmy ponad 90% dokładności. Nasza sieć nie jest przeuczona, różnica 3 % między zbiorem testowym i uczącym jest dobra